# Lia — preparação e adaptação emocional com Qwen3-TTS oficial

Notebook privado para Colab: envia uma gravação longa **uma vez** ao runtime, transcreve localmente, segmenta em recortes de fala, oferece revisão rápida de transcrição/emoção e prepara o conjunto para o fine-tuning oficial single-speaker do Qwen3-TTS Base. Áudio, transcrições, pesos e checkpoints ficam em `/content/lia_qwen_emotion`; nada vai para Drive, Git, API de ASR ou serviço de demonstração.

**O que é automático:** ASR local, cortes em frases de 3–16 s, limpeza/validação, JSONL e preparação de códigos. **Revisão humana pequena, mas necessária:** corrigir transcrições ruins, escolher uma referência limpa e classificar algumas emoções; não existe classificador de emoção confiável o bastante para rotular a voz da dubladora sem escuta.

O fine-tuning upstream aceita apenas `audio`, `text` e `ref_audio`: rótulos de emoção são guardados para auditoria/eval, não entram como controle supervisionado no treino oficial. Depois do fine-tune, o notebook testa o speaker customizado com `instruct` de emoção. A T4 atual pode preparar tudo, mas o trainer oficial exige BF16 + memória maior; por segurança, a etapa de treino é bloqueada automaticamente em T4 (não tenta baixar o modelo nem inicia treino). Usa-se o trainer oficial apenas numa GPU compatível.

Referências: [fine-tuning oficial](https://github.com/QwenLM/Qwen3-TTS/tree/022e286b98fbec7e1e916cb940cdf532cd9f488e/finetuning), [API oficial](https://github.com/QwenLM/Qwen3-TTS).

In [ ]:
# 0. Ambiente local no Colab e dependências de preparação
from pathlib import Path
import os, sys, subprocess, importlib.util, json, csv, hashlib, random, shutil, math, time, re

ROOT = Path('/content/lia_qwen_emotion')
SEG_DIR = ROOT / 'segments'
SOURCE_DIR = ROOT / 'source_audio'
ROOT.mkdir(parents=True, exist_ok=True)
SEG_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

# Instala dependências ausentes no runtime descartável do Colab; Whisper roda em CPU int8.
from importlib.metadata import version, PackageNotFoundError
try:
    fw_version=version('faster-whisper')
except PackageNotFoundError:
    fw_version=None
if fw_version != '1.2.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'faster-whisper==1.2.1'], check=True)
for package, module in [('soundfile','soundfile'), ('ipywidgets','ipywidgets')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', package], check=True)

# faster-whisper 1.2.1 passa metadata_errors a av.open(); PyAV 19 removeu esse argumento.
# Forçamos PyAV 18.1.0 num diretório isolado e antes de importar faster-whisper/av.
PYAV_COMPAT = ROOT / 'pyav_compat'
if not (PYAV_COMPAT / 'av' / '__init__.py').is_file():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir',
                    '--target', str(PYAV_COMPAT), '--no-deps', 'av==18.1.0'], check=True)
sys.path.insert(0, str(PYAV_COMPAT))
# If this cell is rerun after the incompatible PyAV was imported, discard its module references.
for module_name in list(sys.modules):
    if module_name == 'av' or module_name.startswith('av.') or module_name == 'faster_whisper' or module_name.startswith('faster_whisper.'):
        del sys.modules[module_name]
import av
if av.__version__ != '18.1.0':
    raise RuntimeError(f'PyAV incompatível ({av.__version__}); esperado exatamente 18.1.0.')
print('faster-whisper:',version('faster-whisper'),'| PyAV isolado:',av.__version__)

import numpy as np, soundfile as sf, ipywidgets as widgets
from faster_whisper import WhisperModel
from IPython.display import Audio, Markdown, clear_output, display
from google.colab import files, output
output.enable_custom_widget_manager()
import torch
print('GPU disponível:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Pasta privada do runtime:', ROOT)

### Retomar treino em outra GPU sem reenviar o WAV de 13 minutos
Se precisar trocar a T4 por A100 de 40 GB (L4 pode ficar abaixo do limite efetivo de 24 GiB), gere antes o pacote na célula opcional ao final da preparação e baixe-o para o computador. No novo runtime, rode a célula de ambiente, defina `IMPORT_TRAIN_BUNDLE=True` nesta célula e execute-a; depois pule ASR/revisão/preparação e execute diretamente a célula de fine-tuning. **Não use “Run all” no runtime novo.** O ZIP inclui apenas os recortes selecionados, referência e metadados necessários — não inclui a gravação original nem caches Whisper.

In [ ]:
# @title Restaurar dataset privado no runtime de treino (opcional)
IMPORT_TRAIN_BUNDLE = False  # mude para True no novo runtime antes de executar esta célula
if not IMPORT_TRAIN_BUNDLE:
    print('Importação desativada. Mantenha False no runtime atual; o pipeline normal segue abaixo.')
else:
    import zipfile
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Envie um único ZIP lia_qwen_training_bundle.zip.')
    zip_name,data=next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'): raise ValueError('Esperado ZIP do dataset curado.')
    bundle_tmp=ROOT/'incoming_training_bundle.zip'; bundle_tmp.write_bytes(data)
    with zipfile.ZipFile(bundle_tmp) as zf:
        root=ROOT.resolve()
        for item in zf.infolist():
            target=(ROOT/item.filename).resolve()
            if target!=root and root not in target.parents: raise ValueError('Caminho inseguro no ZIP; extração bloqueada.')
        zf.extractall(ROOT)
    manifest_path=ROOT/'training_bundle_manifest.json'
    if not manifest_path.is_file(): raise ValueError('Manifest do pacote ausente; não é um pacote deste notebook.')
    manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('format')!='lia-qwen-training-bundle-v1': raise ValueError('Versão de pacote incompatível.')
    for required in ['train_raw.jsonl','holdout_raw.jsonl','review.csv','review_approval.json','emotion_audit.csv','speaker_reference.wav']:
        if not (ROOT/required).is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente no pacote: {required}')
    approval=json.loads((ROOT/'review_approval.json').read_text(encoding='utf-8'))
    if approval.get('review_csv_sha256')!=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest():
        raise ValueError('Hash do review.csv não confere com a aprovação salva.')
    for split in ['train_raw.jsonl','holdout_raw.jsonl']:
        for line in (ROOT/split).read_text(encoding='utf-8').splitlines():
            row=json.loads(line)
            if not Path(row['audio']).is_file() or not Path(row['ref_audio']).is_file():
                raise FileNotFoundError(f'Áudio ausente ao restaurar {split}: {row.get("audio")}')
    print('Dataset curado restaurado e validado em',ROOT)
    print('Pule as células de ASR/segmentação/revisão e execute diretamente a célula “Fine-tuning oficial”.')

## 1. Envie o áudio longo e gere os recortes/transcrições
Envie um único arquivo da dubladora (`wav`, `mp3`, `m4a`, `flac` ou `ogg`). O áudio é decodificado e transcrito no próprio runtime Colab, sem API hospedada. O Whisper gera timestamps; o notebook junta palavras em frases até 14 s, respeitando pausas/pontuação, salva WAVs mono/24 kHz e uma tabela para revisão. Reiniciar o runtime apaga estes arquivos: faça download do ZIP final apenas se quiser guardá-lo no seu computador.

In [ ]:
# 1. Upload, ASR local e segmentação automática
from huggingface_hub import snapshot_download

SOURCE_POINTER=ROOT/'source_audio.json'
if SOURCE_POINTER.is_file():
    stored=json.loads(SOURCE_POINTER.read_text(encoding='utf-8'))
    source=ROOT/stored['relative_path']
    if not source.is_file(): raise FileNotFoundError(f'Áudio original registrado não existe: {source}')
elif globals().get('UPLOAD_DONE') and globals().get('source') and Path(globals()['source']).is_file():
    # Adota o upload feito pelo notebook anterior, sem pedir o WAV novamente.
    source=Path(globals()['source'])
    SOURCE_POINTER.write_text(json.dumps({'relative_path':str(source.relative_to(ROOT))},ensure_ascii=False),encoding='utf-8')
else:
    old_audio=sorted(p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower() in {'.wav','.mp3','.m4a','.flac','.ogg'})
    if len(old_audio)==1:
        source=old_audio[0]
        SOURCE_POINTER.write_text(json.dumps({'relative_path':str(source.relative_to(ROOT))},ensure_ascii=False),encoding='utf-8')
    elif len(old_audio)>1:
        raise RuntimeError(f'Encontrei vários áudios na pasta raiz: {[p.name for p in old_audio]}. Selecione apenas o original e execute novamente.')
    else:
        print('Envie o WAV/MP3 ORIGINAL da Lia (não TTS gerado).')
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError('Envie exatamente um arquivo de áudio.')
        name,data=next(iter(uploaded.items()))
        source=SOURCE_DIR/Path(name).name
        source.write_bytes(data)
        SOURCE_POINTER.write_text(json.dumps({'relative_path':str(source.relative_to(ROOT))},ensure_ascii=False),encoding='utf-8')
    UPLOAD_DONE=True

if source.suffix.lower() not in {'.wav','.mp3','.m4a','.flac','.ogg'}:
    raise ValueError('Formato não reconhecido.')
sha = hashlib.sha256(source.read_bytes()).hexdigest()
manifest = ROOT / 'asr_manifest.json'
if manifest.exists():
    old = json.loads(manifest.read_text(encoding='utf-8'))
else:
    old = {}

cache_ok=False
if old.get('sha256')==sha and old.get('asr_revision')=='536b0662742c02347bc0e980a01041f333bce120' and (ROOT/'review.csv').exists():
    with (ROOT/'review.csv').open(encoding='utf-8-sig',newline='') as f:
        cached_rows=list(csv.DictReader(f))
    cache_ok=bool(cached_rows) and all((SEG_DIR/f"{r.get('id','')}.wav").is_file() for r in cached_rows)
if cache_ok:
    print('ASR e recortes em cache (hash, modelo e WAVs íntegros); pulando transcrição.')
else:
    # Decode once to mono float32 at Qwen's 24 kHz training rate.
    pieces=[]
    with av.open(str(source)) as container:
        stream=container.streams.audio[0]
        resampler=av.AudioResampler(format='flt', layout='mono', rate=24000)
        for frame in container.decode(stream):
            for fr in resampler.resample(frame):
                pieces.append(fr.to_ndarray().reshape(-1).astype(np.float32))
        for fr in resampler.resample(None):
            pieces.append(fr.to_ndarray().reshape(-1).astype(np.float32))
    if not pieces: raise ValueError('Não consegui decodificar o áudio.')
    wave=np.concatenate(pieces)
    duration=len(wave)/24000
    print(f'Áudio: {duration/60:.2f} min, mono/24 kHz; Whisper local small em CPU int8.')

    # Model revision is fixed for reproducibility; files are cached in /content only.
    asr_dir=snapshot_download('Systran/faster-whisper-small', revision='536b0662742c02347bc0e980a01041f333bce120')
    whisper=WhisperModel(asr_dir, device='cpu', compute_type='int8')
    segments, info=whisper.transcribe(str(source), language='pt', beam_size=5,
                                      vad_filter=True, word_timestamps=True,
                                      vad_parameters={'min_silence_duration_ms':300})
    segments=list(segments)
    words=[]
    for seg in segments:
        if seg.words:
            for w in seg.words:
                txt=(w.word or '').strip()
                if txt: words.append({'start':float(w.start),'end':float(w.end),'text':txt})
        elif seg.text.strip():
            words.append({'start':float(seg.start),'end':float(seg.end),'text':seg.text.strip()})
    if not words: raise RuntimeError('Whisper não encontrou fala; confirme o arquivo.')

    # Split on pause/punctuation/length. Then merge fragments that are too short.
    groups=[]; cur=[]; start=None; previous_end=None
    for w in words:
        if cur and (w['start']-previous_end > 0.62 or w['start']-start >= 13.5):
            groups.append(cur); cur=[]; start=None
        if start is None: start=w['start']
        cur.append(w); previous_end=w['end']
        if w['text'].endswith(('.', '?','!','…')) or w['end']-start >= 13.5:
            groups.append(cur); cur=[]; start=None; previous_end=None
    if cur: groups.append(cur)
    merged=[]
    for g in groups:
        if merged and (g[-1]['end']-g[0]['start'] < 2.6) and (g[-1]['end']-merged[-1][0]['start'] <= 15.5) and g[0]['start']-merged[-1][-1]['end'] < 0.9:
            merged[-1].extend(g)
        else: merged.append(g)
    rows=[]
    for g in merged:
        a=max(0.0,g[0]['start']-0.06); b=min(duration,g[-1]['end']+0.10)
        if b-a < 2.6 or b-a > 16.0: continue
        text=re.sub(r'\s+([,.;:!?…])',r'\1',' '.join(w['text'] for w in g)).strip()
        if not text: continue
        idx=len(rows)+1; clip_id=f'lia_{idx:04d}'
        path=SEG_DIR/f'{clip_id}.wav'
        i0=max(0,int(a*24000)); i1=min(len(wave),int(b*24000))
        clip=wave[i0:i1]
        if len(clip) < int(2.6*24000): continue
        sf.write(path,clip,24000,subtype='PCM_16')
        rows.append({'id':clip_id,'start_s':round(a,3),'end_s':round(b,3),'duration_s':round((b-a),3),
                     'text':text,'include':'yes','emotion':'nao_rotulado','reference':'no'})
    if len(rows)<8: raise RuntimeError(f'Só {len(rows)} recortes utilizáveis; confira a gravação/transcrição.')
    with (ROOT/'review.csv').open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
    manifest.write_text(json.dumps({'sha256':sha,'source_name':source.name,'duration_s':duration,
        'asr_model':'Systran/faster-whisper-small','asr_revision':'536b0662742c02347bc0e980a01041f333bce120',
        'detected_language':info.language,'language_probability':info.language_probability,'segments':len(rows)},ensure_ascii=False,indent=2),encoding='utf-8')
    print(f'Idioma detectado: {info.language} ({info.language_probability:.2f}); {len(rows)} recortes criados.')
    print('Transcrição bruta guardada em review.csv; o Whisper pode errar palavras, pontuação e nomes.')

print('Arquivos em',ROOT,'| CSV para revisão:',ROOT/'review.csv')

## 2. Escute, corrija e marque emoção rapidamente
O painel percorre os recortes: ouça, corrija a transcrição, marque se entra no treino, classifique a emoção **quando identificável** e escolha exatamente uma referência de speaker limpa. `Não rotulado` pode ficar para falas que você não quer usar como exemplo emocional; os rótulos são metadados de controle de qualidade, não serão tratados como verdade automática. Para não enviesar a emoção-alvo, priorize gravações limpas e realmente interpretadas pela dubladora (não leitura neutra de texto).

In [ ]:
# 2. Revisor interativo: cada gravação é reproduzida localmente no notebook
LABELS=['nao_rotulado','neutro','alegre','comemorativa','triste','irritada','calma','assertiva','outra']
review_path=ROOT/'review.csv'
with review_path.open(encoding='utf-8-sig',newline='') as f:
    review_rows=list(csv.DictReader(f))
if not review_rows: raise ValueError('review.csv vazio; execute a célula anterior.')
for r in review_rows:
    if r.get('emotion') not in LABELS: r['emotion']='nao_rotulado'

idx=widgets.IntSlider(value=0,min=0,max=len(review_rows)-1,description='Recorte',continuous_update=False,layout=widgets.Layout(width='80%'))
transcript=widgets.Textarea(description='Texto',layout=widgets.Layout(width='100%',height='80px'))
include=widgets.Checkbox(description='Usar no treino',indent=False)
emotion=widgets.Dropdown(options=LABELS,description='Emoção')
reference=widgets.Checkbox(description='Usar como referência fixa da voz',indent=False)
audio_box=widgets.Output(); status=widgets.HTML()
prev_btn=widgets.Button(description='Anterior'); save_btn=widgets.Button(description='Salvar'); next_btn=widgets.Button(description='Salvar e próximo',button_style='primary')

def persist_review():
    with review_path.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(review_rows[0])); w.writeheader(); w.writerows(review_rows)

def save_form():
    r=review_rows[idx.value]
    r['text']=transcript.value.strip(); r['include']='yes' if include.value else 'no'
    r['emotion']=emotion.value
    if reference.value:
        for other in review_rows: other['reference']='no'
        r['reference']='yes'
    else:
        r['reference']='no'
    persist_review()

def load_form(change=None):
    r=review_rows[idx.value]
    transcript.value=r['text']; include.value=(r['include'].lower()=='yes')
    emotion.value=r['emotion'] if r['emotion'] in LABELS else 'nao_rotulado'
    reference.value=(r['reference'].lower()=='yes')
    with audio_box:
        clear_output(wait=True)
        display(Audio(filename=str(SEG_DIR/f"{r['id']}.wav"),autoplay=False))
    status.value=f"<b>{idx.value+1}/{len(review_rows)} — {r['id']} — {r['duration_s']} s — {r['start_s']}–{r['end_s']} s</b>"

def save_click(_): save_form(); status.value += ' — salvo'
def next_click(_):
    save_form()
    if idx.value<idx.max: idx.value+=1
    else: status.value='Fim da lista — alterações salvas.'
def prev_click(_):
    save_form()
    if idx.value>0: idx.value-=1
idx.observe(load_form,names='value'); save_btn.on_click(save_click); next_btn.on_click(next_click); prev_btn.on_click(prev_click)
load_form()
display(widgets.VBox([status,idx,audio_box,transcript,widgets.HBox([include,emotion,reference]),widgets.HBox([prev_btn,save_btn,next_btn])]))
print('Use “Salvar e próximo”. O CSV é atualizado a cada recorte; pode parar e retomar sem perder o que já foi salvo.')

## 3. Validar e preparar dataset oficial
O trainer oficial recomenda o mesmo `ref_audio` em todas as linhas para estabilidade. Escolha como referência um recorte limpo (sem música/ruído/corte), preferencialmente neutro; não escolha automaticamente a primeira fala. O notebook valida os textos, remove itens desmarcados, reserva um holdout, mantém rótulos de emoção em `emotion_audit.csv` e cria o JSONL oficial (`audio`, `text`, `ref_audio`).

In [ ]:
# 3. Preparação determinística; não altera nem apaga o WAV original
with (ROOT/'review.csv').open(encoding='utf-8-sig',newline='') as f:
    rows=list(csv.DictReader(f))
selected=[r for r in rows if r.get('include','').lower()=='yes']
selected=[r for r in selected if r.get('text','').strip() and (SEG_DIR/f"{r['id']}.wav").is_file()]
if len(selected)<8: raise ValueError(f'Apenas {len(selected)} exemplos incluídos; revise/transcreva ao menos 8.')
print('Revise as transcrições/emoções abaixo antes de autorizar qualquer fine-tuning:')
for r in selected:
    print(f"{r['id']} | {r['duration_s']} s | {r.get('emotion','nao_rotulado')} | {r['text']}")
review_ok=input('Depois de conferir/corrigir review.csv e escutar os recortes incluídos, digite REVISEI para preparar o dataset: ').strip()
if review_ok!='REVISEI': raise RuntimeError('Revisão não confirmada; nenhum dataset de treino foi liberado.')
refs=[r for r in selected if r.get('reference','').lower()=='yes']
if len(refs)!=1: raise ValueError(f'Marque exatamente uma referência de speaker; selecionadas: {len(refs)}.')
ref_row=refs[0]
ref_src=SEG_DIR/f"{ref_row['id']}.wav"
ref_audio=ROOT/'speaker_reference.wav'
shutil.copy2(ref_src,ref_audio)

# Fixed seed and shuffled held-out utterances; never train on holdout.
random.Random(20261003).shuffle(selected)
holdout_n=max(1,round(len(selected)*0.15))
if len(selected)-holdout_n<5: holdout_n=1
holdout=selected[:holdout_n]; train=selected[holdout_n:]

def write_jsonl(path, items):
    with path.open('w',encoding='utf-8') as f:
        for r in items:
            item={'audio':str((SEG_DIR/f"{r['id']}.wav").resolve()),'text':r['text'].strip(),
                  'ref_audio':str(ref_audio.resolve()),'language':'Portuguese','emotion_audit':r.get('emotion','nao_rotulado')}
            f.write(json.dumps(item,ensure_ascii=False)+'\n')
write_jsonl(ROOT/'train_raw.jsonl',train)
write_jsonl(ROOT/'holdout_raw.jsonl',holdout)
with (ROOT/'emotion_audit.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['id','text','emotion','duration_s','split']);w.writeheader()
    for split,items in [('train',train),('holdout',holdout)]:
        for r in items:w.writerow({'id':r['id'],'text':r['text'],'emotion':r.get('emotion','nao_rotulado'),'duration_s':r['duration_s'],'split':split})
print(f'Treino: {len(train)} recortes / {sum(float(r["duration_s"]) for r in train)/60:.1f} min; holdout: {len(holdout)}.')
print('Referência fixa:',ref_row['id'],ref_row['duration_s'],'s | rótulos:',dict((x,sum(r.get('emotion')==x for r in selected)) for x in LABELS))
approval={'review_csv_sha256':hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest(),
          'source_sha256':sha,'approved_at':time.strftime('%Y-%m-%dT%H:%M:%S%z')}
(ROOT/'review_approval.json').write_text(json.dumps(approval,ensure_ascii=False,indent=2),encoding='utf-8')
print('JSONL/CSV privados prontos em',ROOT,'| revisão confirmada e vinculada ao hash do CSV.')

### Exportar o dataset curado antes de trocar de runtime (opcional)
A sessão Colab pode apagar `/content` ao trocar de GPU. Esta célula empacota apenas os WAVs de treino/holdout, `speaker_reference.wav`, transcrições, rótulos e aprovação de revisão. O WAV original de 12,83 min e caches de modelos ficam de fora. Baixe o ZIP pelo navegador e carregue-o no novo runtime usando a célula de restauração acima.

In [ ]:
# @title Baixar pacote do dataset para continuar em uma GPU maior
EXPORT_TRAIN_BUNDLE = False  # mude para True quando for trocar de runtime
if not EXPORT_TRAIN_BUNDLE:
    print('Exportação desativada. Ative somente antes de trocar/perder o runtime atual.')
else:
    import zipfile
    required=[ROOT/'train_raw.jsonl',ROOT/'holdout_raw.jsonl',ROOT/'review.csv',ROOT/'review_approval.json',ROOT/'emotion_audit.csv',ROOT/'speaker_reference.wav']
    missing=[p.name for p in required if not p.is_file()]
    if missing: raise FileNotFoundError(f'Execute a preparação/revisão primeiro; faltam: {missing}')
    review_hash=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest()
    approval_path=ROOT/'review_approval.json'
    if approval_path.is_file():
        approval=json.loads(approval_path.read_text(encoding='utf-8'))
        if approval.get('review_csv_sha256')!=review_hash:
            raise RuntimeError('review.csv mudou após a aprovação; repita a validação antes de exportar.')
    else:
        print('Confirme que já ouviu/revisou os recortes e corrigiu as transcrições no painel.')
        answer=input('Digite REVISEI para autorizar o pacote de treino: ').strip()
        if answer!='REVISEI': raise RuntimeError('Pacote não exportado; revisão não confirmada.')
        asr_manifest=json.loads((ROOT/'asr_manifest.json').read_text(encoding='utf-8')) if (ROOT/'asr_manifest.json').is_file() else {}
        approval={'review_csv_sha256':review_hash,'source_sha256':asr_manifest.get('sha256','unknown'),
                  'approved_at':time.strftime('%Y-%m-%dT%H:%M:%S%z'),'legacy_session_confirmed':True}
        approval_path.write_text(json.dumps(approval,ensure_ascii=False,indent=2),encoding='utf-8')
    audio_files=set(required)
    for split in ['train_raw.jsonl','holdout_raw.jsonl']:
        for line in (ROOT/split).read_text(encoding='utf-8').splitlines():
            row=json.loads(line)
            for key in ['audio','ref_audio']:
                path=Path(row[key]).resolve()
                if not path.is_file() or not path.is_relative_to(ROOT.resolve()):
                    raise ValueError(f'Caminho de áudio inválido/fora do workspace: {path}')
                audio_files.add(path)
    manifest={'format':'lia-qwen-training-bundle-v1','created_at':time.strftime('%Y-%m-%dT%H:%M:%S%z'),
              'train_rows':sum(1 for _ in (ROOT/'train_raw.jsonl').open(encoding='utf-8')),
              'holdout_rows':sum(1 for _ in (ROOT/'holdout_raw.jsonl').open(encoding='utf-8')),
              'contains_original_recording':False,'review_csv_sha256':approval['review_csv_sha256']}
    manifest_path=ROOT/'training_bundle_manifest.json'
    manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    bundle=ROOT/'lia_qwen_training_bundle.zip'
    with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_STORED) as zf:
        for path in sorted(audio_files): zf.write(path,path.relative_to(ROOT))
        zf.write(manifest_path,manifest_path.relative_to(ROOT))
    print(f'Pacote pronto: {bundle.name}; {bundle.stat().st_size/1024**2:.1f} MiB; sem WAV original/cache.')
    files.download(str(bundle))

## 4. Fine-tuning oficial e teste de instrução emocional (só em GPU compatível)
A etapa abaixo usa os scripts oficiais no commit fixado `022e286…` e o modelo `Qwen/Qwen3-TTS-12Hz-0.6B-Base`. Para evitar compilar FlashAttention, faz um patch local documentado para usar PyTorch SDPA. O modelo e o tokenizer são baixados por revisões fixas. Só inicia se a GPU tem BF16 e pelo menos 24 GiB; numa T4, prepara os dados e para aqui sem baixar os pesos de treino. A saída fica no runtime Colab. Duas épocas são um smoke test conservador, não prova de qualidade. Depois gera a mesma frase com instrução alegre e neutra para avaliação auditiva.

In [ ]:
# 4. Treino oficial condicionado a hardware compatível (T4 será bloqueada com explicação)
RUN_TRAINING = True
QWEN_COMMIT='022e286b98fbec7e1e916cb940cdf532cd9f488e'
MODEL_ID='Qwen/Qwen3-TTS-12Hz-0.6B-Base'
MODEL_REV='5d83992436eae1d760afd27aff78a71d676296fc'
TOKENIZER_ID='Qwen/Qwen3-TTS-Tokenizer-12Hz'
TOKENIZER_REV='7dd38ad4e9bad454aae9cd937d0cd577604fe229'
approval_path=ROOT/'review_approval.json'
if not approval_path.is_file(): raise RuntimeError('Treino bloqueado: primeiro revise os recortes e confirme na etapa 3.')
approval=json.loads(approval_path.read_text(encoding='utf-8'))
if approval.get('review_csv_sha256')!=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest():
    raise RuntimeError('review.csv mudou após a confirmação; repita a preparação/revisão antes do treino.')
if not RUN_TRAINING:
    print('RUN_TRAINING=False: dataset preparado; treino não iniciado.')
elif not torch.cuda.is_available():
    print('Treino bloqueado: não há GPU CUDA. Dataset permanece pronto em',ROOT)
else:
    props=torch.cuda.get_device_properties(0)
    vram=props.total_memory/1024**3
    bf16=bool(torch.cuda.is_bf16_supported())
    print(f'GPU {props.name}; VRAM {vram:.1f} GiB; BF16 {bf16}')
    if not bf16 or vram < 24:
        print('Treino oficial bloqueado com segurança: upstream exige BF16 e full SFT; esta GPU não cumpre BF16 + 24 GiB. Nada foi baixado nem treinado. Os JSONL e WAVs estão prontos em /content/lia_qwen_emotion.')
    else:
        # Validate current pinned official trainer inputs before pulling base weights.
        repo=ROOT/'Qwen3-TTS-official'
        if not repo.exists():
            subprocess.run(['git','clone','https://github.com/QwenLM/Qwen3-TTS.git',str(repo)],check=True)
        subprocess.run(['git','-C',str(repo),'fetch','origin',QWEN_COMMIT],check=True)
        subprocess.run(['git','-C',str(repo),'checkout','--force',QWEN_COMMIT],check=True)
        trainer=repo/'finetuning'/'sft_12hz.py'
        original=trainer.read_text(encoding='utf-8')
        if 'attn_implementation="flash_attention_2"' not in original:
            raise RuntimeError('O trainer oficial fixado mudou; interrompido para revisão, sem patch automático.')
        patched=original.replace('attn_implementation="flash_attention_2"','attn_implementation="sdpa"')
        trainer.write_text(patched,encoding='utf-8')
        diff_path=ROOT/'official_trainer_sdpa.patch.txt'
        diff_path.write_text('Patch único e local: attn_implementation flash_attention_2 -> sdpa.\nCommit upstream: '+QWEN_COMMIT+'\n',encoding='utf-8')
        # Keep helper libraries isolated under /content; reuse Colab's CUDA Torch/Torchaudio.
        if shutil.disk_usage(ROOT).free < 15*1024**3:
            raise RuntimeError('Menos de 15 GiB livres para snapshots/checkpoints; libere espaço antes de iniciar o treino.')
        deps=ROOT/'qwen_deps'; deps.mkdir(exist_ok=True)
        deps_marker=deps/'.ready_qwen_train_deps'
        if not deps_marker.exists():
            subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(deps),
                            'numpy>=2.1,<2.4','transformers==4.57.3','accelerate==1.12.0','librosa==0.11.0','soundfile','onnxruntime','einops','tensorboard','huggingface_hub==0.36.2'],check=True)
            deps_marker.write_text('transformers 4.57.3; accelerate 1.12.0; numpy <2.4\n',encoding='utf-8')
        (deps/'sitecustomize.py').write_text('try:\n import transformers.utils.import_utils as _u\n _u._torchvision_available=False\nexcept Exception:\n pass\n',encoding='utf-8')
        train_env=os.environ.copy(); train_env['PYTHONPATH']=os.pathsep.join([str(deps),str(repo),train_env.get('PYTHONPATH','')])
        from huggingface_hub import snapshot_download
        model_path=snapshot_download(MODEL_ID,revision=MODEL_REV)
        tokenizer_path=snapshot_download(TOKENIZER_ID,revision=TOKENIZER_REV)
        # Prepare audio codes with the official tokenizer utility.
        prep=repo/'finetuning'/'prepare_data.py'
        subprocess.run([sys.executable,str(prep),'--device','cuda:0','--tokenizer_model_path',tokenizer_path,
                        '--input_jsonl',str(ROOT/'train_raw.jsonl'),'--output_jsonl',str(ROOT/'train_with_codes.jsonl')],check=True,env=train_env)
        out=ROOT/'checkpoints'
        out.mkdir(exist_ok=True)
        cmd=[sys.executable,'-u',str(trainer),'--init_model_path',model_path,'--output_model_path',str(out),
             '--train_jsonl',str(ROOT/'train_with_codes.jsonl'),'--batch_size','1','--lr','2e-6',
             '--num_epochs','2','--speaker_name','lia_emocional']
        print('Iniciando SFT single-speaker oficial; logs aparecem em tempo real. WAVs e modelo continuam no Colab.',flush=True)
        proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout: print(line,end='',flush=True)
        code=proc.wait()
        if code: raise subprocess.CalledProcessError(code,cmd)
        print('Fine-tuning terminou. Checkpoints:',*[str(p) for p in sorted(out.glob('checkpoint-*'))])

In [ ]:
# 5. Inferência controlada: mesmo texto, speaker treinado, instrução alegre vs neutra
# Execute depois de um fine-tuning concluído; em T4 esta etapa fica indisponível sem checkpoint.
from pathlib import Path
checkpoints=sorted((ROOT/'checkpoints').glob('checkpoint-*')) if (ROOT/'checkpoints').exists() else []
if not checkpoints:
    print('Nenhum checkpoint de fine-tuning encontrado. Se a GPU era T4, isso é esperado; dataset está salvo em',ROOT)
else:
    import soundfile as sf
    from IPython.display import Audio,display
    import sys
    sys.path.insert(0,str(ROOT/'qwen_deps'))
    sys.path.insert(0,str(ROOT/'Qwen3-TTS-official'))
    import transformers.utils.import_utils as _hf_import_utils
    _hf_import_utils._torchvision_available=False
    from qwen_tts import Qwen3TTSModel
    import torch
    checkpoint=str(checkpoints[-1])
    model=Qwen3TTSModel.from_pretrained(checkpoint,device_map='cuda:0',dtype=torch.bfloat16,attn_implementation='sdpa')
    text='Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
    prompts=[('alegre','Fale em português brasileiro com alegria calorosa e entusiasmo genuíno, como quem comemora uma vitória; natural, sem gritar.'),
             ('neutro','Fale em português brasileiro de forma neutra, natural e conversacional.')]
    for label,instruct in prompts:
        wavs,sr=model.generate_custom_voice(text=text,language='Portuguese',speaker='lia_emocional',instruct=instruct)
        path=ROOT/f'lia_{label}_finetuned.wav';sf.write(path,wavs[0],sr)
        print(label, path);display(Audio(filename=str(path)))

### O que o resultado vai permitir concluir
- Se a voz/sotaque ficam mais consistentes e o prompt `alegre` diferencia a emoção no speaker fine-tuned, ouvimos as duas saídas e seguimos para validação em outras frases.
- Se treino/inferência piorarem voz ou sotaque, os originais e o modelo base continuam intactos; não se substitui nada automaticamente.
- O holdout fica separado para preservar uma avaliação não vista durante o fine-tuning. Não integre ainda ao Lia-Omni: primeiro valide voz e controle emocional como especialistas.
- Em T4, este notebook automatiza upload, transcrição local, segmentação, revisão e preparação do dataset. A limitação de treino é do trainer oficial (BF16/full fine-tune), não do corpus; não tenta trocar precisão ou instalar um trainer de terceiros às cegas.